# 3 — Choose a View and evaluate one loaded resonance

> **Lesson focus**
>
> **Learn:** derive an analysis View without changing the physical
> circuit. **Run:** retain one node and evaluate its loaded diagonal
> root. **Inspect:** the immutable View lineage and typed root evidence.
> **Status:** `CONVERGING` scaffold.

## Derive the analysis boundary without changing the circuit

One physical circuit can support several questions. Lesson 2 used the
complete one-Port response; this lesson asks about the authored
resonator node instead. SCNSim keeps those questions separate from the
circuit itself:

``` text
CircuitPlan → CircuitRun → original View → ReductionPipeline
            → child View → Spec and operation → typed Result
```

| Object | Responsibility |
|------------------------------------|------------------------------------|
| `CircuitPlan` | The one physical topology and its components, nodes, ground, and Ports. |
| `run.original` | The sealed Plan’s zero-reduction root `NetworkViewRef`. |
| `ReductionPipeline` | An ordered declaration of effective topology, coordinates, and final boundary. |
| child `NetworkViewRef` | An immutable, lazy analysis lineage—not a new circuit or a Result. |
| Spec and `run.evaluate()` | The physical question and the terminal operation that executes it. |
| typed Result | The materialized answer and its evidence surfaces. |

A pipeline has three ordered jobs. `ptc()` can first change the
effective analysis topology; lesson 10 applies it to nonloading probes.
`transform_pair()` can then change the coordinate basis; lesson 11
derives common and differential coordinates. Terminal `retain()` selects
the final external boundary, and this lesson uses that simplest
pipeline.

Declaring a pipeline does not compile, solve, or write the workspace.
`reduce()` returns a new child Ref without mutating its parent or the
Plan, so the same root Ref can branch into independent response and
quantity Views. A View is backend-neutral: it declares the analysis
lineage, while the later terminal operation chooses and checks the
compatible execution path.

In [ ]:
from fixtures.primitive_resonator import build_primitive_resonator
from scnsim import CircuitRun, ReductionPipeline

fixture = build_primitive_resonator()
run = CircuitRun(plan=fixture.plan, workspace="workspaces/primitive-course")

response_view = run.original
quantity_pipeline = ReductionPipeline().retain(
    fixture.resonator_node
)
quantity_view = response_view.reduce(quantity_pipeline)

`retain(fixture.resonator_node)` keeps that Public coordinate as the
selected boundary. Every other coordinate receives zero external
current, but its components and existing loads remain in the effective
circuit. In particular, the 50-ohm Port from lesson 1 is not removed.
Retaining a coordinate is not deleting components, slicing rows from a
matrix, or changing the schematic.

`solve()` needs a Port-realizable View because it materializes an S/Y/Z
network response. `evaluate()` may instead use a Public node View such
as `quantity_view` to request a typed Direct quantity. A terminal
operation checks that capability before executing.

## Evaluate the loaded resonance through that View

`DiagonalRootSpec` asks for the diagonal root associated with the
retained resonator coordinate. `root_hint` initializes the deterministic
complex-Newton basin at 6.0 GHz; it is not the answer, target, search
window, nearest-root request, or proof that the complete spectrum has
only one root.

In [ ]:
from scnsim import DiagonalRootSpec, units as u

root_spec = DiagonalRootSpec(
    coordinate=fixture.resonator_node,
    root_hint=6.0 * u.GHz,
)
root = run.evaluate(quantity_view, root_spec)

The 50-ohm Port remains in the selected effective circuit, so this is a
loaded root. It is neither the uncoupled LC formula nor the minimum of
lesson 2’s S-response samples. Unlike `solve()`, `evaluate()`
materializes only the requested physical quantity and its branch
evidence.

In [ ]:
root.frequency
root.linewidth
root.slope
root.show()

`frequency`, `linewidth`, and `slope` belong to the same typed Result.
`show()` presents that existing Result and never evaluates again.

[Previous](02_solve_direct.qmd) · [Course map](../../docs/index.qmd) ·
[Next: tune and verify a primitive resonance](04_optimize_primitive.qmd)
· [Concept: network View
lineage](../../docs/concepts/compilation-coordinates-and-network-views.qmd#network-view-lineage)